## **Part 1 - Setup**

In [1]:
!pip install dask
!pip install psutil

In [2]:
import pandas as pd
import numpy as np
import dask.dataframe as dd

import json
import time
import psutil

from multiprocessing import Pool
from concurrent.futures import ThreadPoolExecutor

from google.colab import drive

In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


## **Part 2 - Load Existing Dataset**

In [4]:
with open(
    "/content/drive/MyDrive/DataDucks Project P1/raw_data.json",
    encoding="utf-8"
) as f:
    data = json.load(f)

print("Total Records:", len(data))

Total Records: 30834


In [5]:
df = pd.DataFrame(data)

print(df.shape)

df.head()

(30834, 5)


,title,url,category,date,excerpt
0,No more RM1 interbank ATM fee from 1st July 20...,https://soyacincau.com/2026/06/15/banks-malays...,Digital Life,"June 15, 2026","While most Malaysians have gone cashless, here..."
1,"Perodua QV-E is now priced from RM63,499: Also...",https://soyacincau.com/2026/06/15/perodua-qve-...,EV,"June 15, 2026",Perodua QV-E can now be obtained from as low a...
2,"Meta offers Instagram Plus, WhatsApp Plus and ...",https://soyacincau.com/2026/06/15/meta-instagr...,Apps,"June 15, 2026",Meta has finally rolled out its paid subscript...
3,iCaur Malaysia upgrades EV battery warranty to...,https://soyacincau.com/2026/06/14/icaur-malays...,Cars,"June 14, 2026",iCaur Malaysia has announced enhanced warranty...
4,"Vivo Y31s Malaysia: 6,500mAh battery, IP69+ du...",https://soyacincau.com/2026/06/14/vivo-y31s-5g...,Featured,"June 14, 2026",Vivo Malaysia has introduced the new Vivo Y31s...


## **Part 3 - Benchmark Sequential Version**

In [6]:
def clean_sequential(df):

    df = df.copy()

    df["title"] = (
        df["title"]
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

    df["category"] = (
        df["category"]
        .str.strip()
        .str.title()
    )

    df["excerpt"] = (
        df["excerpt"]
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

    df["url"] = df["url"].str.strip()

    df["date"] = pd.to_datetime(
        df["date"],
        errors="coerce"
    )

    df = df.dropna(subset=["date"])

    df["date"] = (
        df["date"]
        .dt.strftime("%Y-%m-%d")
    )

    df = df[
        df["url"].str.startswith("http")
    ]

    return df

In [7]:
cpu_before = psutil.cpu_percent(interval=None)
memory_before = psutil.virtual_memory().percent

start = time.time()

sequential_df = clean_sequential(df)

sequential_time = time.time() - start

cpu_after = psutil.cpu_percent(interval=None)
memory_after = psutil.virtual_memory().percent

sequential_cpu = cpu_after
sequential_memory = memory_after

print("Sequential Time:", round(sequential_time, 4), "seconds")
print("CPU Usage:", sequential_cpu, "%")
print("Memory Usage:", sequential_memory, "%")

Sequential Time: 0.6444 seconds
CPU Usage: 57.7 %
Memory Usage: 9.1 %


## **Part 4 - Multithreading Version**

In [8]:
def clean_chunk(chunk):

    chunk = chunk.copy()

    chunk["title"] = (
        chunk["title"]
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

    chunk["category"] = (
        chunk["category"]
        .str.strip()
        .str.title()
    )

    chunk["excerpt"] = (
        chunk["excerpt"]
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

    chunk["url"] = chunk["url"].str.strip()

    chunk["date"] = pd.to_datetime(
        chunk["date"],
        errors="coerce"
    )

    chunk = chunk.dropna(subset=["date"])

    chunk["date"] = (
        chunk["date"]
        .dt.strftime("%Y-%m-%d")
    )

    chunk = chunk[
        chunk["url"].str.startswith("http")
    ]

    return chunk

In [9]:
chunks = np.array_split(df, 4)

/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [10]:
cpu_before = psutil.cpu_percent(interval=None)
memory_before = psutil.virtual_memory().percent

start = time.time()

with ThreadPoolExecutor(max_workers=4) as executor:

    results = list(
        executor.map(
            clean_chunk,
            chunks
        )
    )

thread_df = pd.concat(results)

thread_time = time.time() - start

cpu_after = psutil.cpu_percent(interval=None)
memory_after = psutil.virtual_memory().percent

thread_cpu = cpu_after
thread_memory = memory_after

print("Multithreading Time:", round(thread_time, 4), "seconds")
print("CPU Usage:", thread_cpu, "%")
print("Memory Usage:", thread_memory, "%")

Multithreading Time: 0.6806 seconds
CPU Usage: 56.3 %
Memory Usage: 9.1 %


## **Part 5 - Multiprocessing Version**

In [11]:
chunks = np.array_split(df, 4)

/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [12]:
cpu_before = psutil.cpu_percent(interval=None)
memory_before = psutil.virtual_memory().percent

start = time.time()

with Pool(4) as pool:

    results = pool.map(
        clean_chunk,
        chunks
    )

process_df = pd.concat(results)

process_time = time.time() - start

cpu_after = psutil.cpu_percent(interval=None)
memory_after = psutil.virtual_memory().percent

process_cpu = cpu_after
process_memory = memory_after

print("Multiprocessing Time:", round(process_time, 4), "seconds")
print("CPU Usage:", process_cpu, "%")
print("Memory Usage:", process_memory, "%")

Multiprocessing Time: 0.9123 seconds
CPU Usage: 93.4 %
Memory Usage: 9.3 %


## **Part 6 - Dask Version**

In [13]:
ddf = dd.from_pandas(
    df,
    npartitions=4
)

In [14]:
cpu_before = psutil.cpu_percent(interval=None)
memory_before = psutil.virtual_memory().percent

start = time.time()

ddf["title"] = (
    ddf["title"]
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

ddf["category"] = (
    ddf["category"]
    .str.strip()
    .str.title()
)

ddf["excerpt"] = (
    ddf["excerpt"]
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

dask_df = ddf.compute()

dask_time = time.time() - start

cpu_after = psutil.cpu_percent(interval=None)
memory_after = psutil.virtual_memory().percent

dask_cpu = cpu_after
dask_memory = memory_after

print("Dask Time:", round(dask_time, 4), "seconds")
print("CPU Usage:", dask_cpu, "%")
print("Memory Usage:", dask_memory, "%")

Dask Time: 0.3456 seconds
CPU Usage: 79.4 %
Memory Usage: 9.7 %


## **Part 7 - Save Results**

In [15]:
process_df.to_csv(
    "/content/drive/MyDrive/DataDucks Project P1/optimized_data.csv",
    index=False
)

print("Optimized dataset saved")

Optimized dataset saved


## **Part 8 - Generate Performance Metrics**

In [16]:
results = pd.DataFrame({

    "Method":[
        "Sequential",
        "Multithreading",
        "Multiprocessing",
        "Dask"
    ],

    "Execution Time (s)":[
        sequential_time,
        thread_time,
        process_time,
        dask_time
    ],

    "CPU Usage (%)":[
        sequential_cpu,
        thread_cpu,
        process_cpu,
        dask_cpu
    ],

    "Memory Usage (%)":[
        sequential_memory,
        thread_memory,
        process_memory,
        dask_memory
    ]
})

results

,Method,Execution Time (s),CPU Usage (%),Memory Usage (%)
0,Sequential,0.644428,57.7,9.1
1,Multithreading,0.680555,56.3,9.1
2,Multiprocessing,0.912282,93.4,9.3
3,Dask,0.345628,79.4,9.7


In [17]:
results["Throughput (records/sec)"] = (
    len(df)
    /
    results["Execution Time (s)"]
)

results

,Method,Execution Time (s),CPU Usage (%),Memory Usage (%),Throughput (records/sec)
0,Sequential,0.644428,57.7,9.1,47847.099601
1,Multithreading,0.680555,56.3,9.1,45307.147894
2,Multiprocessing,0.912282,93.4,9.3,33798.777629
3,Dask,0.345628,79.4,9.7,89211.639319


In [18]:
results.to_csv(
    "/content/drive/MyDrive/DataDucks Project P1/performance_metrics.csv",
    index=False
)

print("Performance metrics saved")

Performance metrics saved


In [19]:
results["Execution Time (s)"] = (
    results["Execution Time (s)"]
    .round(4)
)

results["CPU Usage (%)"] = (
    results["CPU Usage (%)"]
    .round(2)
)

results["Memory Usage (%)"] = (
    results["Memory Usage (%)"]
    .round(2)
)

results["Throughput (records/sec)"] = (
    results["Throughput (records/sec)"]
    .round(2)
)

results

,Method,Execution Time (s),CPU Usage (%),Memory Usage (%),Throughput (records/sec)
0,Sequential,0.6444,57.7,9.1,47847.10
1,Multithreading,0.6806,56.3,9.1,45307.15
2,Multiprocessing,0.9123,93.4,9.3,33798.78
3,Dask,0.3456,79.4,9.7,89211.64
